### Import Dependencies

In [1]:
import openai
import pandas as pd

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

### Create Qdrant collection for hybrid search

In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [4]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01-hybrid-search",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams(modifier=Modifier.IDF)
    }
)

True

In [5]:
qdrant_client.create_payload_index(
    collection_name="Amazon-items-collection-01-hybrid-search",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding Functions

In [6]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [7]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Read the sampled dataset with Amazon inventory data

In [8]:
df_items = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [9]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,AMAZON FASHION,"Bosttor Bluetooth Beanie Hat with Light, Headl...",4.5,2342,"[100% Acrylic, Elastic closure, Hand Wash Only...",[],26.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Beanie Hat with Bluetooth Headphon...,Bosttor,"[Electronics, Headphones, Earbuds & Accessorie...","{'Department': 'unisex-adult', 'Date First Ava...",B0BH41HYFZ,NaN,NaN,NaN
1,All Electronics,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",4.2,148,[【USB or USB C to Ethernet Adapter】The Etherne...,[],14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aceele,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.54 x 0.98 x 0.71 inc...,B0BKPB2YQ9,NaN,NaN,NaN
2,All Electronics,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,4.2,3331,[📍3-Port HDMI Switch: This aluminum HDMI switc...,[],18.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Darren reviews VWRHAR 3-1 splitter...,VWRHar,"[Electronics, Home Audio, Home Audio Accessori...",{'Package Dimensions': '6.1 x 4.21 x 0.94 inch...,B09MM5QT3R,NaN,NaN,NaN
3,All Electronics,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",3.4,125,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Burxoe,[],{'Package Dimensions': '3.15 x 3.07 x 2.52 inc...,B09Q5TNDHY,NaN,NaN,NaN
4,Cell Phones & Accessories,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",4.7,335,[【Compatible Airpods 1/2 】This case designed f...,[1],11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'airpod cases', 'url': 'https://www...",UGUHY,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.44 x 4.61 x 1.57 inc...,B0BZJKX7MS,NaN,NaN,NaN


### Preprocess title and features

In [11]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [12]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [13]:
df_items["preprocessed_description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [14]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,preprocessed_description,image
0,AMAZON FASHION,"Bosttor Bluetooth Beanie Hat with Light, Headl...",4.5,2342,"[100% Acrylic, Elastic closure, Hand Wash Only...",[],26.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Beanie Hat with Bluetooth Headphon...,Bosttor,"[Electronics, Headphones, Earbuds & Accessorie...","{'Department': 'unisex-adult', 'Date First Ava...",B0BH41HYFZ,NaN,NaN,NaN,"Bosttor Bluetooth Beanie Hat with Light, Headl...",https://m.media-amazon.com/images/I/51b7qcj8dZ...
1,All Electronics,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",4.2,148,[【USB or USB C to Ethernet Adapter】The Etherne...,[],14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aceele,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.54 x 0.98 x 0.71 inc...,B0BKPB2YQ9,NaN,NaN,NaN,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",https://m.media-amazon.com/images/I/41WsGRr-3T...
2,All Electronics,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,4.2,3331,[📍3-Port HDMI Switch: This aluminum HDMI switc...,[],18.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Darren reviews VWRHAR 3-1 splitter...,VWRHar,"[Electronics, Home Audio, Home Audio Accessori...",{'Package Dimensions': '6.1 x 4.21 x 0.94 inch...,B09MM5QT3R,NaN,NaN,NaN,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,https://m.media-amazon.com/images/I/41fsjaknf1...
3,All Electronics,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",3.4,125,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Burxoe,[],{'Package Dimensions': '3.15 x 3.07 x 2.52 inc...,B09Q5TNDHY,NaN,NaN,NaN,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",https://m.media-amazon.com/images/I/51YFypeuZh...
4,Cell Phones & Accessories,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",4.7,335,[【Compatible Airpods 1/2 】This case designed f...,[1],11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'airpod cases', 'url': 'https://www...",UGUHY,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.44 x 4.61 x 1.57 inc...,B0BZJKX7MS,NaN,NaN,NaN,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",https://m.media-amazon.com/images/I/41V7Wi2ckb...


In [15]:
list(df_items["preprocessed_description"].items())[0]

(0,
 'Bosttor Bluetooth Beanie Hat with Light, Headlamp Cap with Headphones and Built-in Speaker Mic, Gifts for Men Women Teen 100% Acrylic Elastic closure Hand Wash Only 【Upgraded Bluetooth Beanie】Bosttor upgraded Bluetooth wireless technology offer much stable and strong connection, support music and calling, easy and fast to pair with your devices. Maximum transmission distance up to 33 feet. Built-in Stereo Speakers & Mic, enhance your music listening experience. 【Long Working Time Music Hat】Built with easy-access USB charging port, simply charge it via the included USD cable. It takes 1-2 hours to get full charged. The battery offers continuous working hours up to 10 hours. Perfect for camping, hiking, skiing, hunting, jogging, cycling, dog walking, auto repair, etc. 【LED Headlight Light Up Your Way】Hands-free LED light is rechargeable and removable. You can remove it to charge by laptop, power bank, socket, car charger, etc. The 4 LED lights can light up to 30 feet away. It point

In [16]:
list(df_items["image"].items())[0]

(0, 'https://m.media-amazon.com/images/I/51b7qcj8dZL._AC_.jpg')

In [17]:
df_data_to_embed = df_items[["preprocessed_description", "image", "rating_number", "price", "average_rating", "parent_asin"]]

In [18]:
df_data_to_embed.head()

,preprocessed_description,image,rating_number,price,average_rating,parent_asin
0,"Bosttor Bluetooth Beanie Hat with Light, Headl...",https://m.media-amazon.com/images/I/51b7qcj8dZ...,2342,26.99,4.5,B0BH41HYFZ
1,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",https://m.media-amazon.com/images/I/41WsGRr-3T...,148,14.99,4.2,B0BKPB2YQ9
2,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,https://m.media-amazon.com/images/I/41fsjaknf1...,3331,18.98,4.2,B09MM5QT3R
3,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",https://m.media-amazon.com/images/I/51YFypeuZh...,125,NaN,3.4,B09Q5TNDHY
4,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",https://m.media-amazon.com/images/I/41V7Wi2ckb...,335,11.99,4.7,B0BZJKX7MS


In [19]:
data_to_embed = df_data_to_embed.to_dict(orient="records")

In [20]:
data_to_embed

[{'preprocessed_description': 'Bosttor Bluetooth Beanie Hat with Light, Headlamp Cap with Headphones and Built-in Speaker Mic, Gifts for Men Women Teen 100% Acrylic Elastic closure Hand Wash Only 【Upgraded Bluetooth Beanie】Bosttor upgraded Bluetooth wireless technology offer much stable and strong connection, support music and calling, easy and fast to pair with your devices. Maximum transmission distance up to 33 feet. Built-in Stereo Speakers & Mic, enhance your music listening experience. 【Long Working Time Music Hat】Built with easy-access USB charging port, simply charge it via the included USD cable. It takes 1-2 hours to get full charged. The battery offers continuous working hours up to 10 hours. Perfect for camping, hiking, skiing, hunting, jogging, cycling, dog walking, auto repair, etc. 【LED Headlight Light Up Your Way】Hands-free LED light is rechargeable and removable. You can remove it to charge by laptop, power bank, socket, car charger, etc. The 4 LED lights can light up 

In [21]:
len(data_to_embed)

1000

In [22]:
text_to_embed = [item["preprocessed_description"] for item in data_to_embed]

In [23]:
text_to_embed

['Bosttor Bluetooth Beanie Hat with Light, Headlamp Cap with Headphones and Built-in Speaker Mic, Gifts for Men Women Teen 100% Acrylic Elastic closure Hand Wash Only 【Upgraded Bluetooth Beanie】Bosttor upgraded Bluetooth wireless technology offer much stable and strong connection, support music and calling, easy and fast to pair with your devices. Maximum transmission distance up to 33 feet. Built-in Stereo Speakers & Mic, enhance your music listening experience. 【Long Working Time Music Hat】Built with easy-access USB charging port, simply charge it via the included USD cable. It takes 1-2 hours to get full charged. The battery offers continuous working hours up to 10 hours. Perfect for camping, hiking, skiing, hunting, jogging, cycling, dog walking, auto repair, etc. 【LED Headlight Light Up Your Way】Hands-free LED light is rechargeable and removable. You can remove it to charge by laptop, power bank, socket, car charger, etc. The 4 LED lights can light up to 30 feet away. It points di

In [24]:
embeddings = get_embeddings_batch(text_to_embed)

Processed 100 of 1000
Processed 200 of 1000
Processed 300 of 1000
Processed 400 of 1000
Processed 500 of 1000
Processed 600 of 1000
Processed 700 of 1000
Processed 800 of 1000
Processed 900 of 1000
Processed 1000 of 1000


In [25]:
len(embeddings)

1000

In [26]:
pointstructs = []
i=1
for embedding, data in zip(embeddings, data_to_embed):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding,
                "bm25": Document(
                    text=data["preprocessed_description"],
                    model="qdrant/bm25"
                )
            },
            payload=data
        )
    )
    i += 1

In [ ]:
pointstructs[0].vector

PointStruct(id=1, vector={'text-embedding-3-small': [0.0118865966796875, 0.010284423828125, 0.001575469970703125, -0.0296478271484375, -0.071044921875, -0.08697509765625, -0.0194854736328125, 0.01470184326171875, 0.0391845703125, -0.07940673828125, -0.00606536865234375, 0.0215911865234375, -0.04974365234375, -0.005542755126953125, 0.01103973388671875, 0.0234832763671875, -0.0701904296875, -0.033050537109375, -0.042572021484375, 0.0285797119140625, -0.0287322998046875, 0.01416015625, 0.00426483154296875, -0.007625579833984375, 0.01715087890625, 0.046478271484375, -0.0203857421875, 0.0215911865234375, 0.05206298828125, 0.001247406005859375, -0.04241943359375, -0.0261077880859375, 0.026702880859375, -0.025299072265625, -0.0080413818359375, 0.0039043426513671875, 0.014251708984375, -0.061553955078125, 0.0335693359375, -0.01151275634765625, -0.0034351348876953125, -0.0303192138671875, 0.06793212890625, 0.001667022705078125, 0.0121307373046875, -0.012176513671875, -0.03009033203125, 0.018753

In [29]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[0:500],
    wait=True
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [30]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[500:],
    wait=True
)

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

### Hybrid Retrieval

In [31]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [32]:
results = retrieve_data("Can I get a tablet?", k=20)

In [33]:
results

{'retrieved_context_ids': ['B0BN58Z4YX',
  'B0B157WDDJ',
  'B0BSD3QK7M',
  'B09RHF4L45',
  'B0B159KDFP',
  'B0BHVH4D37',
  'B0B58CPFWY',
  'B0C7DCS2KW',
  'B0BCFYCXRH',
  'B0B6ZZH83Y',
  'B0BGHBL86V',
  'B09WR36NK8',
  'B0BT83RRJ2',
  'B0BWRZ1HRD',
  'B09XQMRYBJ',
  'B09SZNLXJQ',
  'B0B2KJKT86',
  'B07Y36DDYM',
  'B0BZCM9CBR',
  'B09RN3KN5C'],
 'retrieved_context': ['ASWINN Tablet Tripod Stand, Gooseneck 65" Height Adjustable Tablet Stand Floor with 360° Rotating Tripod Mount Suitable for iPhone,Tablet,Kindle and All 4.5-12.9 Inch Tablet and Phone (Black) 【Free Your Hands】: With this gooseneck arm tablet tripod stand, you can find a more comfortable way to use tablet & Phone in bed or sofa in winter. Release your hands while make your leisure time more enjoyable，the best Christmas gift or New Year gifts for wife,husband,elders,friend or colleague. 【Versatile Gooseneck Tablet Tripod Stand】This ergonomic gooseneck tablet holder suits most tablets and phones. you can easily get your desir

### Hybrid Search with weighted RRF

In [34]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [35]:
results = retrieve_data("Can I get a tablet?", k=20)

In [36]:
results

{'retrieved_context_ids': ['B0B157WDDJ',
  'B0BN58Z4YX',
  'B0BSD3QK7M',
  'B0B159KDFP',
  'B09RHF4L45',
  'B0BGHBL86V',
  'B0BHVH4D37',
  'B0C7DCS2KW',
  'B09WR36NK8',
  'B0BCFYCXRH',
  'B0BT83RRJ2',
  'B0B58CPFWY',
  'B0BWRZ1HRD',
  'B0B6ZZH83Y',
  'B07Y36DDYM',
  'B09RN3KN5C',
  'B0BNKF9J6J',
  'B0B96TTN6T',
  'B08SC3C9MC',
  'B09PYV6B4B'],
 'retrieved_context': ["G-TiDE Kids Tablet, 7 inch Tablet for Kids, 32GB+2GB Kids Learning Tablet, 5MP Dual Camera HD, Parental Control App- KLAP, Toddler Tablet Case, WiFi Tablets Shoulder Straps, Pink 🤗【Explore More Fun on Klap】G-TiDE Klap kids tablet is designed for learning and playing. This tablet for kids offers various creative contents such as brain training, painting, gaming, kids TV, etc. Learning while playing is better for kids to know the world. With a 32GB bigger storage, which can be extended to 128GB (micro SD card not included), this kids tablet is perfect for children aged 3-7 years old. You could get more educational apps from 